In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import re
from scipy.interpolate import interp1d

In [ ]:
#Set plot parameters
#plt.rc(‘text’, usetex=True)
plt.rcParams['font.size'] = 28
plt.rcParams['axes.labelsize'] = 28
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['xtick.major.size'] = 8
plt.rcParams['ytick.major.size'] = 8
plt.rcParams['xtick.minor.size'] = 4
plt.rcParams['ytick.minor.size'] = 4
plt.rcParams['xtick.major.width'] =2
plt.rcParams['ytick.major.width'] =2
plt.rcParams['xtick.minor.width'] =2
plt.rcParams['ytick.minor.width'] =2
plt.rcParams['xtick.labelsize'] = 22
plt.rcParams['ytick.labelsize'] = 22

In [ ]:
## useful function
def ABmag_to_Flam(magAB,lamb=1.):
    term = 0.4*(8.9-magAB)-4.-np.log10(3.34)-2.*np.log10(lamb)
    Flam=10.**(term)
    
    return(Flam)

In [ ]:
## function to estimate EW from pa_alpha magnitude 

def find_ew_pa_mag(mag_lowf,mag_linef,mag_highf,plam_lowf=15010.,plam_linef=18740.,plam_highf=19900.,bw_lowf=3170.,bw_linef=240.,bw_highf=4630.):
    
    ## mag_lowf: magnitude of the lower wl cont filter (F150W)
    ## mag_linef: magnitude of the line fiter (F187N)
    ## mag_highf: magnitude of the higher wl cont filter (F200W)
    ## plam_lowf: lambda pivot of the lower wl cont filter (F150W)
    ## plam_linef: lambda pivot of the line fiter (F187N)
    ## plam_highf: lambda pivot of the higher wl cont filter (F200W)
    ## bw_lowf=3360: bandwidth of the lower wl cont filter (F150W)
    ## bw_linef=200: bandwidth of the line fiter (F187N)
    ## bw_highf=4730: bandwidth of the higher wl cont filter (F200W) 
    
    
    
    flux_lowf   = ABmag_to_Flam(mag_lowf,lamb=plam_lowf)
    flux_linef = ABmag_to_Flam(mag_linef,lamb=plam_linef)
    flux_highf = ABmag_to_Flam(mag_highf,lamb=plam_highf)
    
    logflux_lowf,logflux_highf = np.log10(flux_lowf),np.log10(flux_highf)
    f1 = interp1d([plam_lowf,plam_highf],[logflux_lowf,logflux_highf])
    
    flux_linefc = 10.**f1(plam_linef)
    flux_linefl = flux_linef-flux_linefc
    ew1 = (flux_linefl)/flux_linefc*bw_linef
    ew=ew1

    for jj in range(5):
        cont_highf = flux_highf-flux_linefl*bw_linef/bw_highf
        logcont_highf = np.log10(cont_highf)
        f1 = interp1d([plam_lowf,plam_highf],[logflux_lowf,logcont_highf])
        flux_linefc = 10.**f1(plam_linef)
        flux_linefl = flux_linef-flux_linefc
        ew = (flux_linefl)/flux_linefc*bw_linef
    
    return(ew)

In [ ]:
def read_trials(file_path): ## help function: read the slug trials 
    
    data = []
    #print('starting reading the file')
    with open(file_path, 'r') as file:
        lines = file.readlines()
        
        
        column_names = re.split(r'\s+', lines[0].strip())
        num_columns = len(column_names)
       
        
        ### we skip the first three rows
        for line in lines[3:]:
            stripped_line = line.strip()
            
            # Skip separator rows (long lines of dashes)
            if all(char == '-' for char in stripped_line):  # Check if the line is all dashes
                continue
                
            line_noempty = re.sub(r'\s{21}', ' NaN ', stripped_line) ## Slug reserves 21 chars: so this is to treat the empty entries
            
            
            data.append(re.split(r'\s+',line_noempty ))
    df = pd.DataFrame(data, columns=column_names)
    return df

In [ ]:
path2out = '/project/galaxies/tjuchau/projects/EW_vsAge/slug_tests/'

In [ ]:
masses = ['5e2Msun', '1e3Msun', '3e3Msun','5e3Msun', '1e4Msun']

masskey = '1e4Msun'

fn = f'clusters_{masskey}_cluster_phot.txt'
lib = read_trials(path2out+fn)
lib['UniqueID'] = pd.to_numeric(lib['UniqueID'], errors='coerce') ## we convert to numbers
lib['Time'] = pd.to_numeric(lib['Time'], errors='coerce') ## we convert to numbers
lib['QH0'] = pd.to_numeric(lib['QH0'], errors='coerce') ## we convert to numbers
lib['JWST_NC_F150W_n'] = pd.to_numeric(lib['JWST_NC_F150W_n'], errors='coerce') ## we convert to numbers
lib['JWST_NC_F200W_n'] = pd.to_numeric(lib['JWST_NC_F200W_n'], errors='coerce') ## we convert to numbers
lib['JWST_NC_F187N_n'] = pd.to_numeric(lib['JWST_NC_F187N_n'], errors='coerce') ## we convert to numbers

In [ ]:
ew_mod = np.zeros(len(lib))

for i in range(len(lib)):
    ew_i = find_ew_pa_mag(mag_lowf=lib['JWST_NC_F150W_n'].values[i], mag_linef=lib['JWST_NC_F187N_n'].values[i], mag_highf=lib['JWST_NC_F200W_n'].values[i])
    ew_mod[i] = ew_i

In [ ]:
##make tables with only relevant info for each specific mass 

new_modtab = {
    
    'UniqueID': lib['UniqueID'].values,
    'Age': lib['Time'].values,
    'QH0': lib['QH0'].values,
    'EW_PaA': ew_mod,
    
}

models_ew = pd.DataFrame(new_modtab)
outname = '/Users/alpe2383/Desktop/FEAST/Models/EWmodels_SLUG_0.73cov_n100_M10000_Zsol.txt'

#models_ew.to_csv(outname, index = False, sep = '\t', na_rep='NaN')

## Analysis of the EWs

In [ ]:
## now we open the SLUG tables we created in the cell above

slug_500 = pd.read_csv('/Users/alpe2383/Desktop/FEAST/Models/EWmodels_SLUG_0.73cov_n100_M500_Zsol.txt', sep = '\s+')
slug_1000 = pd.read_csv('/Users/alpe2383/Desktop/FEAST/Models/EWmodels_SLUG_0.73cov_n100_M1000_Zsol.txt', sep = '\s+')
slug_3000 = pd.read_csv('/Users/alpe2383/Desktop/FEAST/Models/EWmodels_SLUG_0.73cov_n100_M3000_Zsol.txt', sep = '\s+')
slug_5000 = pd.read_csv('/Users/alpe2383/Desktop/FEAST/Models/EWmodels_SLUG_0.73cov_n100_M5000_Zsol.txt', sep = '\s+')
slug_10000 = pd.read_csv('/Users/alpe2383/Desktop/FEAST/Models/EWmodels_SLUG_0.73cov_n100_M10000_Zsol.txt', sep = '\s+')



In [ ]:
fig = plt.figure(figsize =(10,5))
ax = fig.add_subplot(111)
for key, grp in slug_1000.groupby('UniqueID'):
    ax.plot(grp['Age']/1e6, grp['EW_PaA'], color = 'grey', alpha = 0.01)
             
pivot_df = slug_1000.pivot(index='Age', columns='UniqueID', values='EW_PaA')
median_ew = pivot_df.median(axis=1)

ax.plot(median_ew.index / 1e6, median_ew.values, color='black', linewidth=2, label='Median')
ax.text(7,4000,r'1000 M$_{\odot}$')





ax.set_xlabel('Age [Myr]')
ax.set_ylabel(r'EW$_{{\rm Pa}\alpha}$')


    

ax.set_xlim(0.5,10.5)



plt.show()